# Symposium - Exploratory Data Analysis

This notebook is a *guided tour* of the DWH after you've run the ETL and at least one debate. It exercises:

- **Pandas** (data manipulation)
- **Matplotlib + Seaborn** (visualisation)
- **Scikit-Learn** (clustering + PCA)
- **NumPy + SciPy** (numeric ops)

Run cells top-to-bottom.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font='serif')

from symposium.dwh import Warehouse
wh = Warehouse()
counts = wh.counts()
counts

## 1. Chunk features by topic

In [ ]:
chunks = wh.query('SELECT c.*, d.topic_key, d.title FROM fact_chunk c JOIN dim_document d USING(doc_id)')
chunks.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=chunks, x='topic_key', y='evidence_density', ax=ax[0])
ax[0].set_title('Evidence density by topic')
sns.boxplot(data=chunks, x='topic_key', y='narrative_density', ax=ax[1])
ax[1].set_title('Narrative density by topic')
plt.tight_layout()

## 2. Persona affinity correlation

In [ ]:
score_cols = [c for c in chunks.columns if c.startswith('score_')]
corr = chunks[score_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('How correlated are persona affinities?')

## 3. PCA of chunk feature space

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

feat_cols = ['date_density','evidence_density','narrative_density','counter_density','reading_ease','avg_word_len']
X = StandardScaler().fit_transform(chunks[feat_cols].values)
pca = PCA(n_components=2).fit(X)
xy = pca.transform(X)
chunks_pca = chunks.assign(x=xy[:,0], y=xy[:,1])
sns.scatterplot(data=chunks_pca, x='x', y='y', hue='topic_key', alpha=0.6)
plt.title(f'PCA of chunk features (explained variance {pca.explained_variance_ratio_.round(2)})')

## 4. Debate stats (if any debates have been run)

In [ ]:
turns = wh.query('SELECT * FROM fact_turn')
print(len(turns), 'turns recorded')
if not turns.empty:
    sns.countplot(data=turns, x='persona_key')
    plt.title('Turns per persona')

In [ ]:
wh.close()